# Section 4 — Higher-Order Taylor Methods

[← Euler Error](03_euler_error.ipynb) · [Module 3](README.md)

Euler's method keeps only the first derivative term. Higher-order Taylor methods keep more terms:

\[y(t+h)=y(t)+h y'(t)+\frac{h^2}{2}y''(t)+\frac{h^3}{6}y^{(3)}(t)+\cdots.\]

### Learning goals
- construct second- and third-order Taylor methods;
- compare their accuracy with Euler's method;
- verify the expected convergence order;
- see why Runge–Kutta methods are attractive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def y1(t, y):
    return t*np.exp(3*t) - 2*y
def y2(t, y):
    return np.exp(3*t)*(1+t) + 4*y
def y3(t, y):
    return np.exp(3*t)*(4+7*t) - 8*y
def exact(t):
    return np.exp(3*t)*(t/5 - 1/25) + np.exp(-2*t)/25

## Taylor 2 and Taylor 3

For the IVP \(y'=te^{3t}-2y,\ y(0)=0\), differentiation gives \(y''=e^{3t}(1+t)+4y\) and \(y^{(3)}=e^{3t}(4+7t)-8y\).

In [ ]:
def euler(t0, y0, h, n_steps):
    t=np.empty(n_steps+1); w=np.empty(n_steps+1); t[0],w[0]=t0,y0
    for n in range(n_steps):
        w[n+1]=w[n]+h*y1(t[n],w[n]); t[n+1]=t[n]+h
    return t,w

def taylor2(t0, y0, h, n_steps):
    t=np.empty(n_steps+1); w=np.empty(n_steps+1); t[0],w[0]=t0,y0
    for n in range(n_steps):
        w[n+1]=w[n]+h*y1(t[n],w[n])+(h**2/2)*y2(t[n],w[n]); t[n+1]=t[n]+h
    return t,w

def taylor3(t0, y0, h, n_steps):
    t=np.empty(n_steps+1); w=np.empty(n_steps+1); t[0],w[0]=t0,y0
    for n in range(n_steps):
        w[n+1]=w[n]+h*y1(t[n],w[n])+(h**2/2)*y2(t[n],w[n])+(h**3/6)*y3(t[n],w[n]); t[n+1]=t[n]+h
    return t,w

In [ ]:
h=0.5; n_steps=2
methods={'Euler':euler,'Taylor 2':taylor2,'Taylor 3':taylor3}
print('method          approximation      exact y(1)        error')
for name,method in methods.items():
    t,w=method(0.0,0.0,h,n_steps)
    err=abs(exact(1.0)-w[-1])
    print(f'{name:10s}      {w[-1]:12.8f}    {exact(1.0):12.8f}   {err:10.6f}')

In [ ]:
t_fine=np.linspace(0,1,400)
fig,ax=plt.subplots(figsize=(9,5))
ax.plot(t_fine, exact(t_fine), linewidth=2, label='exact')
for name,method in methods.items():
    t,w=method(0.0,0.0,h,n_steps); ax.plot(t,w,marker='o',label=name)
ax.set_xlabel('t'); ax.set_ylabel('y'); ax.legend(); ax.grid(True,alpha=0.25)
plt.show()

## Check the convergence order

A method of order \(p\) has approximately \(E(h)\approx C h^p\). Halving \(h\) should reduce the error by about \(2^p\).

In [ ]:
step_sizes=np.array([0.5,0.25,0.125,0.0625])
results={'Taylor 2':[],'Taylor 3':[]}
for h in step_sizes:
    for name,method in [('Taylor 2',taylor2),('Taylor 3',taylor3)]:
        _,w=method(0.0,0.0,h,round(1/h)); results[name].append(abs(exact(1.0)-w[-1]))
for name in results:
    print('\n'+name); previous=None
    for h,err in zip(step_sizes,results[name]):
        p='--' if previous is None else f'{np.log(previous/err)/np.log(2):.4f}'
        print(f'h={h:7.4f}, error={err:12.6e}, observed p={p}')
        previous=err

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
for name,errs in results.items(): ax.loglog(step_sizes,errs,marker='o',label=name)
ax.set_xlabel('step size h'); ax.set_ylabel('absolute error at t=1'); ax.legend(); ax.grid(True,which='both',alpha=0.25)
plt.show()

## Why not keep going with Taylor methods?

Each additional order requires another derivative. For a general ODE, repeated differentiation introduces increasingly complicated chain-rule terms. Runge–Kutta methods seek higher-order accuracy using only carefully chosen evaluations of \(f(t,y)\).

That is where Section 5 will begin.

[← Euler Error](03_euler_error.ipynb) · [↑ Module 3](README.md)